[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/12_linear_attention.ipynb)

# 🔴 Hard: Linear Self-Attention

Implement **Linear Attention** — O(S·D²) instead of O(S²·D), enabling efficient long-sequence processing.

Replace softmax with a **kernel feature map** $\phi$:

$$\text{LinearAttn}(Q,K,V) = \frac{\phi(Q) \left(\phi(K)^T V\right)}{\phi(Q) \cdot \sum \phi(K)}$$

### Feature map
Use $\phi(x) = \text{elu}(x) + 1$ (ensures non-negative features).

### Signature
```python
def linear_attention(Q, K, V):
    # Q: (B, S, D_k), K: (B, S, D_k), V: (B, S, D_v)
    # Returns: (B, S, D_v)
```

### Key insight
Instead of computing the $S \times S$ attention matrix, compute $\phi(K)^T V$ first (a $D_k \times D_v$ matrix), then multiply by $\phi(Q)$.

### Rules
- Must use a feature map (NOT softmax)
- Must be O(S·D²) — should run fast on long sequences
- You **may** use `F.elu`

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.8 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn.functional as F

In [11]:
# ✏️ YOUR IMPLEMENTATION HERE

def linear_attention(Q, K, V):
    # pass  # Replace this
    psi = lambda x: F.elu(x) + 1
    nom = psi(Q).bmm(psi(K).transpose(-2, -1).bmm(V))
    # 1. Sum K over the sequence dimension (dim=1) -> Shape: (B, 1, D_k)
    sum_k = psi(K).sum(dim=1, keepdim=True)

    # 2. Dot product with Q over the feature dimension (dim=-1) -> Shape: (B, S, 1)
    denom = (psi(Q) * sum_k).sum(dim=-1, keepdim=True)

    # 3. Divide (PyTorch broadcasts denom from 1 to D_v)
    return nom / denom

In [12]:
# 🧪 Debug
Q = torch.randn(1, 8, 16)
K = torch.randn(1, 8, 16)
V = torch.randn(1, 8, 32)
out = linear_attention(Q, K, V)
print("Output shape:", out.shape)   # (1, 8, 32)
print("Has NaN?", torch.isnan(out).any().item())

Output shape: torch.Size([1, 8, 32])
Has NaN? False


In [13]:
from torch_judge import check
check('linear_attention')


🧪 Testing: Linear Self-Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (0.7ms)
  ✅ [2/4] No NaN or Inf (15.9ms)
  ✅ [3/4] Gradient flow (19.7ms)
  ✅ [4/4] Runs fast on long sequences (linear complexity) (54.2ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (90.5ms total)
  Progress saved. Run status() to see your dashboard.

